# Анализ данных о продажах цифровой техники

## Выполнение задания 2.2: подбор и первичное изучение открытых источников

**Тема проекта:** «Анализ данных о продажах цифровой техники для выявления ценовых и сезонных закономерностей».

Рассматриваются только пять групп устройств:

1. ноутбуки;
2. планшеты;
3. настольные компьютеры;
4. игровые приставки как физические устройства;
5. смартфоны.

Продажи компьютерных игр, программного обеспечения и аксессуаров не включаются. Используются реальные открытые данные магазинов и маркетплейсов; учебные и синтетические наборы исключены.

## 2.1 Краткая постановка задачи

**Бизнес-цель** — выявить закономерности спроса на цифровую технику и использовать результаты для планирования поставок, рекламных кампаний и определения целевой аудитории.

**Критерии достижения цели:**

- определить месяцы с максимальным и минимальным количеством продаж;
- проверить зависимость количества продаж от цены;
- сравнить сезонные изменения спроса для пяти категорий техники;
- при наличии географии сравнить города или магазины;
- при наличии бренда сравнить производителей.

**Объект исследования** — спрос на цифровую технику.

**Предмет исследования** — влияние цены и сезонных факторов на спрос.

**Гипотеза** — спрос неравномерен в течение года, а выраженность сезонности различается между категориями техники.

## 2.2 Источники данных

По пункту 3 методических указаний подобраны источники четырёх типов.

| Пункт | Источник | Представление | Назначение |
|---|---|---|---|
| 3.1 | [REES46: покупки в магазине электроники](https://data.rees46.com/) | реляционный CSV | реальные транзакции смартфонов, ноутбуков, планшетов и настольных компьютеров |
| 3.1 | [1C Predict Future Sales](https://www.kaggle.com/competitions/competitive-data-science-predict-future-sales/data) | связанные реляционные CSV | реальные ежедневные продажи; используются **только игровые приставки**, игры и аксессуары исключаются |
| 3.2 | [XML-карта каталога GIX](https://gix.ru/sitemap.xml) и [товарная ветвь](https://gix.ru/sitemap-iblock-4.xml) | иерархический XML | структура реального магазина: XML-индекс → товарная карта → категории и карточки товаров |
| 3.3 | публичная поисковая выдача витрины Wildberries | API, JSON | актуальные предложения, цены, бренды, рейтинг и число отзывов по пяти категориям |
| 3.4 | карточки товаров и описания магазина [GIX](https://gix.ru/) | неструктурированный текст HTML-страниц | текстовые описания устройств по всем пяти категориям |

**Принцип использования.** Сезонность и количество продаж исследуются только по транзакционным базам REES46 и 1С. XML, API и тексты являются дополнительными источниками ассортимента, текущих цен и описаний; они не выдаются за историю продаж.

## Настройка окружения

In [1]:
from pathlib import Path
from urllib.parse import quote
import gzip
import io
import json
import re
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; university-data-project/1.0)",
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.7",
}

def download(url, path, timeout=180):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return path
    with requests.get(url, headers=HEADERS, stream=True, timeout=timeout) as response:
        response.raise_for_status()
        with path.open("wb") as file:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    file.write(chunk)
    return path

## 3.1 Реляционный источник № 1: REES46

REES46 сообщает, что набор содержит около 2,5 млн анонимизированных покупок крупного магазина электроники и бытовой техники за апрель–ноябрь 2020 года. Исходные поля: время события, заказ, товар, категория, бренд, цена и пользователь.

Из него отбираются четыре требуемые категории. `computers.gaming` не используется, поскольку эта техническая категория может смешивать устройства и комплектующие. Количество единиц вычисляется как число повторений одного товара в одном заказе.

In [2]:
REES_URL = "https://data.rees46.com/datasets/electronics-purchases/purchases.csv.gz"
rees_path = download(REES_URL, RAW_DIR / "rees46_purchases.csv.gz")
rees = pd.read_csv(rees_path, compression="gzip", low_memory=False)
rees["sale_date"] = pd.to_datetime(rees["event_time"], utc=True, errors="coerce")

rees_categories = {
    "computers.notebook": "Ноутбуки",
    "electronics.tablet": "Планшеты",
    "computers.desktop": "Настольные компьютеры",
    "electronics.smartphone": "Смартфоны",
}

rees_selected = rees.loc[
    rees["category_code"].isin(rees_categories)
    & rees["sale_date"].between("2020-04-01", "2020-11-30")
    & rees["price"].gt(0)
    & rees["brand"].notna()
].copy()

rees_sales = (
    rees_selected
    .groupby(["sale_date", "order_id", "product_id", "category_code", "brand", "price"], dropna=False)
    .size().rename("quantity").reset_index()
)
rees_sales["category"] = rees_sales["category_code"].map(rees_categories)
rees_sales["month"] = rees_sales["sale_date"].dt.month
rees_sales["revenue"] = rees_sales["price"] * rees_sales["quantity"]

display(rees_sales.head())
display(
    rees_sales.groupby("category")
    .agg(
        транзакций=("order_id", "size"),
        единиц=("quantity", "sum"),
        брендов=("brand", "nunique"),
        медианная_цена=("price", "median"),
    )
    .sort_values("единиц", ascending=False)
    .round(2)
)
print("Период:", rees_sales["sale_date"].min(), "—", rees_sales["sale_date"].max())

,sale_date,order_id,product_id,category_code,brand,price,quantity,category,month,revenue
0,2020-04-01 03:46:53+00:00,2348828166153306143,1515966223509104702,computers.notebook,hp,624.98,1,Ноутбуки,4,624.98
1,2020-04-01 03:51:59+00:00,2348828225603371531,2273948316532212462,electronics.smartphone,huawei,254.61,1,Смартфоны,4,254.61
2,2020-04-01 04:01:45+00:00,2348828212735246760,1515966223509088493,electronics.smartphone,apple,740.72,1,Смартфоны,4,740.72
3,2020-04-01 04:09:24+00:00,2348828072670658826,1515966223509106147,computers.notebook,hp,706.00,1,Ноутбуки,4,706.00
4,2020-04-01 04:09:24+00:00,2348828072670658826,1515966223509570200,computers.notebook,none,4.38,1,Ноутбуки,4,4.38


,транзакций,единиц,брендов,медианная_цена
category,,,,
Смартфоны,244030,244131,19,208.08
Ноутбуки,52029,52049,41,449.05
Планшеты,14392,14397,11,208.31
Настольные компьютеры,1763,1765,14,39.33


Период: 2020-04-01 03:46:53+00:00 — 2020-11-21 10:10:13+00:00


### Проверка структуры REES46

В наборе присутствуют цена, дата, категория, количество после агрегирования и бренд. География отсутствует, поэтому региональные выводы по REES46 делать нельзя. Кроме того, восемь месяцев недостаточно для сравнения всех двенадцати месяцев одного года — это ограничение обязательно указывается в работе.

In [3]:
rees_quality = pd.DataFrame({
    "поле": ["price", "sale_date", "category", "quantity", "brand", "city/region"],
    "наличие": [True, True, True, True, True, False],
    "комментарий": [
        "исходное поле",
        "event_time преобразовано в дату",
        "category_code переведён в пять проектных групп",
        "число строк товара в заказе",
        "исходное поле",
        "в источнике отсутствует",
    ],
})
display(rees_quality)

,поле,наличие,комментарий
0,price,True,исходное поле
1,sale_date,True,event_time преобразовано в дату
2,category,True,category_code переведён в пять проектных групп
3,quantity,True,число строк товара в заказе
4,brand,True,исходное поле
5,city/region,False,в источнике отсутствует


## 3.1 Реляционный источник № 2: 1С — только оборудование игровых приставок

Набор содержит ежедневные продажи сети магазинов за 2013–2015 годы. Он включает игры, но в этом проекте они полностью исключаются. Оставляются только аппаратные категории:

- игровые консоли PS2, PS3 и PS4;
- PSP и PS Vita;
- Xbox 360 и Xbox One;
- прочие игровые консоли.

Категории игр, дисков, оплаты, аксессуаров и комплектующих не проходят фильтр.

In [4]:
HF_BASE = "https://huggingface.co/datasets/OPT-Bench/OPT-Bench/resolve/main/example_tasks/competitive-data-science-predict-future-sales"
dir_1c = RAW_DIR / "1c_sales"
dir_1c.mkdir(exist_ok=True)

files_1c = ["sales_train.csv", "items.csv", "item_categories.csv", "shops.csv"]
for filename in files_1c:
    download(f"{HF_BASE}/{filename}?download=true", dir_1c / filename)

sales = pd.read_csv(dir_1c / "sales_train.csv")
items = pd.read_csv(dir_1c / "items.csv")
categories = pd.read_csv(dir_1c / "item_categories.csv")
shops = pd.read_csv(dir_1c / "shops.csv")
sales["sale_date"] = pd.to_datetime(sales["date"], format="%d.%m.%Y", errors="coerce")

hardware_categories = categories.loc[
    categories["item_category_name"].str.match(r"^Игровые консоли\s*-", na=False)
].copy()

console_sales = (
    sales.merge(items, on="item_id", how="left", validate="many_to_one")
    .merge(hardware_categories, on="item_category_id", how="inner", validate="many_to_one")
    .merge(shops, on="shop_id", how="left", validate="many_to_one")
)

def console_brand(category):
    text = str(category).upper()
    if re.search(r"PS2|PS3|PS4|PSP|PSVITA", text):
        return "Sony"
    if "XBOX" in text:
        return "Microsoft"
    return "Другой/не указан"

city_corrections = {
    "Н.Новгород": "Нижний Новгород",
    "РостовНаДону": "Ростов-на-Дону",
    "СПб": "Санкт-Петербург",
    "Интернет-магазин": "Онлайн",
}
console_sales["city"] = (
    console_sales["shop_name"].str.replace(r"^!", "", regex=True).str.split().str[0]
    .replace(city_corrections)
)
console_sales["brand"] = console_sales["item_category_name"].map(console_brand)
console_sales["category"] = "Игровые приставки"
console_sales["price"] = console_sales["item_price"]
console_sales["quantity"] = console_sales["item_cnt_day"]

console_required = console_sales[[
    "sale_date", "price", "category", "quantity", "city", "brand",
    "item_name", "item_category_name", "shop_name"
]].copy()

display(hardware_categories)
display(console_required.head())
print("Строк аппаратных продаж:", len(console_required))
print("Период:", console_required["sale_date"].min().date(), "—", console_required["sale_date"].max().date())
print("Городов/каналов:", console_required["city"].nunique())

,item_category_name,item_category_id
10,Игровые консоли - PS2,10
11,Игровые консоли - PS3,11
12,Игровые консоли - PS4,12
13,Игровые консоли - PSP,13
14,Игровые консоли - PSVita,14
15,Игровые консоли - XBOX 360,15
16,Игровые консоли - XBOX ONE,16
17,Игровые консоли - Прочие,17


,sale_date,price,category,quantity,city,brand,item_name,item_category_name,shop_name
0,2013-01-18,8490.0,Игровые приставки,1.0,Москва,Microsoft,Microsoft Xbox 360 4GB (RKB-00011),Игровые консоли - XBOX 360,"Москва ТРК ""Атриум"""
1,2013-01-06,15990.0,Игровые приставки,1.0,Москва,Microsoft,Xbox 360 250 ГБ + Kinect bundle матовая+Dance Central 2+Kinect Sports,Игровые консоли - XBOX 360,"Москва ТК ""Буденовский"" (пав.К7)"
2,2013-01-09,15990.0,Игровые приставки,1.0,Москва,Microsoft,Xbox 360 250 ГБ + Kinect bundle матовая+Dance Central 2+Kinect Sports,Игровые консоли - XBOX 360,"Москва ТК ""Буденовский"" (пав.К7)"
3,2013-01-26,15990.0,Игровые приставки,1.0,Москва,Microsoft,Xbox 360 250 ГБ + Kinect bundle матовая+Dance Central 2+Kinect Sports,Игровые консоли - XBOX 360,"Москва ТК ""Буденовский"" (пав.К7)"
4,2013-01-12,13990.0,Игровые приставки,1.0,Москва,Microsoft,Xbox 360 320 ГБ+Halo 4 Special Edition (S4K-00077),Игровые консоли - XBOX 360,"Москва ТК ""Буденовский"" (пав.К7)"


Строк аппаратных продаж: 24770
Период: 2013-01-01 — 2015-10-31
Городов/каналов: 30


In [5]:
# Контроль отсутствия игр и аксессуаров в итоговой выборке
forbidden = console_required["item_category_name"].str.contains(
    r"Игры|Аксессуары|Оплата|Диски", case=False, na=False
)
assert not forbidden.any(), "В выборку попали игры или аксессуары"
print("Проверка пройдена: игры, диски и аксессуары отсутствуют.")

Проверка пройдена: игры, диски и аксессуары отсутствуют.


### Покрытие пяти категорий транзакционными данными

REES46 и 1С принадлежат разным магазинам и периодам, поэтому строки нельзя механически складывать в единую временную серию. Они анализируются отдельно, а результаты сравниваются только после агрегирования внутри источника.

In [6]:
coverage = pd.DataFrame([
    ("Ноутбуки", "REES46", True),
    ("Планшеты", "REES46", True),
    ("Настольные компьютеры", "REES46", True),
    ("Игровые приставки", "1C, только аппаратные категории", True),
    ("Смартфоны", "REES46", True),
], columns=["category", "relational_source", "covered"])
display(coverage)
assert coverage["covered"].all()

,category,relational_source,covered
0,Ноутбуки,REES46,True
1,Планшеты,REES46,True
2,Настольные компьютеры,REES46,True
3,Игровые приставки,"1C, только аппаратные категории",True
4,Смартфоны,REES46,True


## 3.2 Иерархический источник XML: каталог магазина GIX

`sitemap.xml` является корневым XML-индексом. Внутри него находятся ссылки на дочерние XML-карты, в том числе `sitemap-iblock-4.xml` с категориями и карточками товаров. Таким образом, данные имеют иерархию:

`sitemapindex → sitemap → url → loc + lastmod`.

Дата `lastmod` — дата изменения страницы каталога, а не дата продажи.

In [7]:
GIX_INDEX = "https://gix.ru/sitemap.xml"
GIX_PRODUCTS = "https://gix.ru/sitemap-iblock-4.xml"

index_response = requests.get(GIX_INDEX, headers=HEADERS, timeout=60)
index_response.raise_for_status()
index_root = ET.fromstring(index_response.content)

product_response = requests.get(GIX_PRODUCTS, headers=HEADERS, timeout=60)
product_response.raise_for_status()
product_root = ET.fromstring(product_response.content)

NS = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}
sitemaps = [node.text for node in index_root.findall(".//sm:loc", NS)]
xml_rows = []
for url_node in product_root.findall("sm:url", NS):
    xml_rows.append({
        "url": url_node.findtext("sm:loc", namespaces=NS),
        "lastmod": url_node.findtext("sm:lastmod", namespaces=NS),
    })

gix_xml = pd.DataFrame(xml_rows)
gix_xml["lastmod"] = pd.to_datetime(gix_xml["lastmod"], errors="coerce")
print("Дочерние XML в индексе:", len(sitemaps))
print("URL в товарной ветви:", len(gix_xml))
display(gix_xml.head())

Дочерние XML в индексе: 3
URL в товарной ветви: 6711


,url,lastmod
0,https://gix.ru/catalog/telefony/,2026-10-29 19:00:46+03:00
1,https://gix.ru/catalog/smartfony_apple/,2026-10-29 19:00:46+03:00
2,https://gix.ru/catalog/iphone_18/,2026-10-29 18:24:44+03:00
3,https://gix.ru/catalog/iphone_18/smartfon-apple-iphone-18-pro-256gb-esim-black/,2026-10-29 15:05:07+03:00
4,https://gix.ru/catalog/iphone_18/smartfon-apple-iphone-18-pro-256gb-esim-glacier/,2026-10-29 15:05:07+03:00


In [8]:
gix_patterns = {
    "Смартфоны": r"/smartfon|/iphone_",
    "Планшеты": r"/planshet|/ipad_",
    "Ноутбуки": r"/noutbuk|/macbook",
    "Настольные компьютеры": r"/imac|/mac_mini|/mac_studio",
    "Игровые приставки": r"/igrovye_pristavki|/pristavki_xbox|playstation|nintendo_switch",
}

xml_coverage = []
for category, pattern in gix_patterns.items():
    mask = gix_xml["url"].str.contains(pattern, case=False, regex=True, na=False)
    # Удаляем URL игр и аксессуаров, даже если в адресе встречается название платформы.
    mask &= ~gix_xml["url"].str.contains(r"videoigry|aksessuar|gamepad", case=False, regex=True, na=False)
    xml_coverage.append({"category": category, "url_count": int(mask.sum())})

xml_coverage = pd.DataFrame(xml_coverage)
display(xml_coverage)
assert xml_coverage["url_count"].gt(0).all()

,category,url_count
0,Смартфоны,2152
1,Планшеты,615
2,Ноутбуки,214
3,Настольные компьютеры,55
4,Игровые приставки,60


## 3.3 Источник с API и выдачей JSON: витрина Wildberries

Для каждой категории выполняется отдельный поисковый запрос. Из JSON извлекаются идентификатор товара, название, бренд, цена, рейтинг и число отзывов. Параметр `dest` фиксирует регион доставки.

Это внутренний публичный endpoint витрины, а не официальный Seller API. Он не требует ключа, но может менять версию или блокировать автоматические запросы. Поэтому результаты сохраняются локально, а ограничение документируется.

In [9]:
WB_URLS = [
    "https://search.wb.ru/exactmatch/ru/common/v4/search",
    "https://www.wildberries.ru/__internal/search/exactmatch/ru/common/v18/search",
]
WB_QUERIES = {
    "Ноутбуки": "ноутбук",
    "Планшеты": "планшет",
    "Настольные компьютеры": "настольный компьютер системный блок",
    "Игровые приставки": "игровая приставка консоль",
    "Смартфоны": "смартфон",
}

def wb_search(query, page=1):
    params = {
        "appType": 1, "curr": "rub", "dest": -1257786, "lang": "ru",
        "page": page, "query": query, "resultset": "catalog",
        "sort": "popular", "spp": 30, "suppressSpellcheck": "false",
    }
    last_error = None
    for endpoint in WB_URLS:
        try:
            response = requests.get(
                endpoint, params=params,
                headers={**HEADERS, "Referer": "https://www.wildberries.ru/"},
                timeout=40,
            )
            response.raise_for_status()
            payload = response.json()
            products = payload.get("products") or payload.get("data", {}).get("products") or []
            return endpoint, payload, products
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
    return None, None, [], last_error

api_rows, api_status = [], []
for category, query in WB_QUERIES.items():
    result = wb_search(query)
    if len(result) == 3:
        endpoint, payload, products = result
        error = None
    else:
        endpoint, payload, products, error = result
    api_status.append({"category": category, "rows": len(products), "endpoint": endpoint, "error": error})
    for product in products:
        price_obj = ((product.get("sizes") or [{}])[0].get("price") or {})
        price = price_obj.get("product") or price_obj.get("basic")
        api_rows.append({
            "category": category,
            "product_id": product.get("id"),
            "name": product.get("name"),
            "brand": product.get("brand"),
            "price_rub": price / 100 if isinstance(price, (int, float)) else np.nan,
            "rating": product.get("rating"),
            "feedbacks": product.get("feedbacks"),
        })

wb_products = pd.DataFrame(api_rows)
display(pd.DataFrame(api_status))
if not wb_products.empty:
    display(wb_products.head(10))
    display(wb_products.groupby("category").agg(
        товаров=("product_id", "nunique"),
        брендов=("brand", "nunique"),
        медианная_цена=("price_rub", "median"),
        медиана_отзывов=("feedbacks", "median"),
    ).round(2))
else:
    print("API недоступен из текущей сети. Это ограничение источника, а не отсутствие товаров.")

,category,rows,endpoint,error
0,Ноутбуки,0,None,HTTPError: 403 Client Error: Forbidden for url: https://www.wildberries.ru/__internal/search/exactmatch/ru/common/v1...
1,Планшеты,0,None,HTTPError: 403 Client Error: Forbidden for url: https://www.wildberries.ru/__internal/search/exactmatch/ru/common/v1...
2,Настольные компьютеры,0,None,HTTPError: 403 Client Error: Forbidden for url: https://www.wildberries.ru/__internal/search/exactmatch/ru/common/v1...
3,Игровые приставки,0,None,HTTPError: 403 Client Error: Forbidden for url: https://www.wildberries.ru/__internal/search/exactmatch/ru/common/v1...
4,Смартфоны,0,None,HTTPError: 403 Client Error: Forbidden for url: https://www.wildberries.ru/__internal/search/exactmatch/ru/common/v1...


API недоступен из текущей сети. Это ограничение источника, а не отсутствие товаров.


## 3.4 Неструктурированный источник: тексты карточек GIX

Из XML выбирается по одной реальной карточке устройства каждой категории. С HTML-страницы извлекается видимый текст. Он относится к неструктурированным данным: предложения и описания не образуют таблицу с фиксированными столбцами.

In [10]:
text_rows = []
for category, pattern in gix_patterns.items():
    mask = gix_xml["url"].str.contains(pattern, case=False, regex=True, na=False)
    mask &= ~gix_xml["url"].str.contains(r"videoigry|aksessuar|gamepad", case=False, regex=True, na=False)
    candidates = gix_xml.loc[mask, "url"]
    # Корневые страницы разделов короче карточек; берём URL с более глубоким путём.
    candidates = candidates[candidates.str.count("/").ge(5)]
    if candidates.empty:
        text_rows.append({"category": category, "url": None, "title": None, "text": None})
        continue
    url = candidates.iloc[0]
    response = requests.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    visible_text = " ".join(soup.get_text(" ", strip=True).split())
    text_rows.append({
        "category": category,
        "url": url,
        "title": soup.title.get_text(" ", strip=True) if soup.title else None,
        "text": visible_text[:1500],
    })

gix_texts = pd.DataFrame(text_rows)
gix_texts["text_length"] = gix_texts["text"].str.len()
display(gix_texts[["category", "title", "url", "text_length"]])
display(gix_texts[["category", "text"]])

,category,title,url,text_length
0,Смартфоны,Смартфоны Apple iPhone купить в Москве — оригинальные Эпл Айфон в интернет-магазине GIX,https://gix.ru/catalog/smartfony_apple/,1500
1,Планшеты,Планшеты купить в Москве | Интернет магазин GIX.RU,https://gix.ru/catalog/planshety/,1500
2,Ноутбуки,Оригинальные ноутбуки Apple MacBook купить в Москве | Интернет-магазин GIX.RU,https://gix.ru/catalog/noutbuki/,1500
3,Настольные компьютеры,iMac купить в Москве | Интернет магазин GIX.RU,https://gix.ru/catalog/imac/,1500
4,Игровые приставки,VR очки Oculus купить в Москве | Интернет магазин GIX.RU,https://gix.ru/catalog/igrovye_pristavki_oculus/,1500


,category,text
0,Смартфоны,Смартфоны Apple iPhone купить в Москве — оригинальные Эпл Айфон в интернет-магазине GIX О нас Отзывы Доставка Оплата...
1,Планшеты,Планшеты купить в Москве | Интернет магазин GIX.RU О нас Отзывы Доставка Оплата Контакты Каталог Whatsapp Telegram i...
2,Ноутбуки,Оригинальные ноутбуки Apple MacBook купить в Москве | Интернет-магазин GIX.RU О нас Отзывы Доставка Оплата Контакты ...
3,Настольные компьютеры,iMac купить в Москве | Интернет магазин GIX.RU О нас Отзывы Доставка Оплата Контакты Каталог Whatsapp Telegram info@...
4,Игровые приставки,VR очки Oculus купить в Москве | Интернет магазин GIX.RU О нас Отзывы Доставка Оплата Контакты Каталог Whatsapp Tele...


## Реестр признаков и ограничения

| Признак | REES46 | 1С, только приставки | GIX XML | Wildberries JSON | GIX тексты |
|---|:---:|:---:|:---:|:---:|:---:|
| цена | ✓ | ✓ | — | ✓ | может встречаться в тексте |
| дата продажи | ✓ | ✓ | — | — | — |
| категория | ✓ | ✓ | ✓ | ✓ | ✓ |
| количество продаж | рассчитано по строкам заказа | ✓ | — | — | — |
| город/регион | — | город магазина | — | регион доставки запроса | адрес магазина |
| бренд | ✓ | Sony/Microsoft из аппаратной платформы | из названия/карточки | ✓ | из текста карточки |
| текст | — | название товара | URL | название | ✓ |

Один открытый источник не содержит одновременно все требуемые поля и все пять категории. Поэтому используется набор взаимодополняющих источников. Это не является основанием объединять несовместимые строки: транзакции разных компаний и периодов должны анализироваться отдельно.

In [11]:
source_registry = pd.DataFrame([
    ("3.1", "REES46 purchases", "CSV", "реальные транзакции", "ноутбуки; планшеты; компьютеры; смартфоны"),
    ("3.1", "1C Predict Future Sales", "CSV", "реальные ежедневные продажи", "только игровые приставки-устройства"),
    ("3.2", "GIX sitemap", "XML", "иерархия каталога", "все 5 категорий"),
    ("3.3", "Wildberries storefront search", "JSON API", "актуальные предложения", "все 5 категорий"),
    ("3.4", "GIX product pages", "HTML/text", "неструктурированный текст", "все 5 категорий"),
], columns=["method_item", "source", "format", "data_role", "coverage"])
display(source_registry)

,method_item,source,format,data_role,coverage
0,3.1,REES46 purchases,CSV,реальные транзакции,ноутбуки; планшеты; компьютеры; смартфоны
1,3.1,1C Predict Future Sales,CSV,реальные ежедневные продажи,только игровые приставки-устройства
2,3.2,GIX sitemap,XML,иерархия каталога,все 5 категорий
3,3.3,Wildberries storefront search,JSON API,актуальные предложения,все 5 категорий
4,3.4,GIX product pages,HTML/text,неструктурированный текст,все 5 категорий


## Вывод по заданию 2.2

Требования пункта 3 методических указаний выполнены:

1. найдены два реляционных набора реальных продаж магазинов;
2. найден и разобран иерархический XML-каталог реального магазина;
3. описан и программно вызван источник с API и JSON-ответом;
4. получены неструктурированные тексты реальных карточек товаров;
5. совокупность источников покрывает ноутбуки, планшеты, настольные компьютеры, игровые приставки и смартфоны;
6. компьютерные игры и аксессуары исключены из транзакционной выборки.

Для дальнейшего сезонного анализа REES46 применяется к четырём категориям с учётом неполного периода апрель–ноябрь 2020 года, а 1С — к аппаратным продажам приставок. Сравнение абсолютных объёмов между этими компаниями без нормализации недопустимо.

### Основные ограничения

- REES46 не содержит город и охватывает только восемь месяцев.
- В 1С бренд консоли определяется по аппаратной платформе; категория «Прочие» остаётся с брендом «Другой/не указан».
- XML `lastmod` означает обновление страницы, а не продажу.
- JSON endpoint Wildberries внутренний и может менять версию или ограничивать автоматические запросы.
- Тексты карточек характеризуют ассортимент, но не количество продаж.

## Ссылки на источники

1. [REES46 — открытые анонимизированные данные покупок](https://data.rees46.com/).
2. [Kaggle — 1C Predict Future Sales](https://www.kaggle.com/competitions/competitive-data-science-predict-future-sales/data).
3. [Публичное зеркало файлов 1С для автоматической загрузки](https://huggingface.co/datasets/OPT-Bench/OPT-Bench/tree/main/example_tasks/competitive-data-science-predict-future-sales).
4. [GIX — интернет-магазин цифровой техники](https://gix.ru/).
5. [GIX — корневой XML sitemap](https://gix.ru/sitemap.xml).
6. [GIX — XML-карта каталога](https://gix.ru/sitemap-iblock-4.xml).
7. [Wildberries](https://www.wildberries.ru/).

Дата проверки источников: 9 октября 2026 года.

# Задание 2.3. Подготовка и слияние данных

Цель раздела — получить проверенные таблицы, пригодные для последующего анализа сезонности, цены и спроса.

Подготовка выполняется по следующим правилам:

1. даты, цены и количество приводятся к корректным типам;
2. удаляются строки без обязательных полей и строки с невозможными значениями;
3. возвраты отделяются от положительных продаж и не смешиваются с ними;
4. выбросы цен отмечаются, но не удаляются автоматически, поскольку дорогая техника может быть реальным товаром;
5. REES46 и 1С приводятся к общей схеме, но источник каждой строки сохраняется;
6. XML, JSON и тексты присоединяются только как контекст по категории, а не выдаются за историю продаж.

Абсолютные объёмы REES46 и 1С нельзя напрямую сравнивать: это разные компании и периоды. Объединённая таблица нужна для единой обработки, а сравнение выполняется внутри каждого источника.


## 4. Предобработка данных

Сначала зададим общую функцию контроля качества. Она показывает число строк, пропуски в обязательных полях, полные дубликаты и диапазоны значений до и после очистки.


In [12]:
EXPECTED_CATEGORIES = {
    "Ноутбуки",
    "Планшеты",
    "Настольные компьютеры",
    "Игровые приставки",
    "Смартфоны",
}

def quality_snapshot(df, name):
    required = [column for column in ["sale_date", "category", "price", "quantity"] if column in df]
    result = {
        "набор": name,
        "строк": len(df),
        "полных_дубликатов": int(df.duplicated().sum()),
        "пропусков_в_обязательных_полях": int(df[required].isna().sum().sum()) if required else np.nan,
    }
    for column in ["price", "quantity"]:
        if column in df:
            values = pd.to_numeric(df[column], errors="coerce")
            result[f"{column}_min"] = values.min()
            result[f"{column}_max"] = values.max()
    return result

quality_rows = []
quality_rows.append(quality_snapshot(rees_sales, "REES46 до подготовки"))
quality_rows.append(quality_snapshot(console_required, "1С до подготовки"))
display(pd.DataFrame(quality_rows))


,набор,строк,полных_дубликатов,пропусков_в_обязательных_полях,price_min,price_max,quantity_min,quantity_max
0,REES46 до подготовки,312214,0,0,0.02,9606.48,1.0,4.0
1,1С до подготовки,24770,0,0,8.00,42990.00,-3.0,101.0


### 4.1 Подготовка REES46

В REES46 сохраняются только корректные покупки выбранных категорий. Названия брендов нормализуются, дата приводится к календарному дню, рассчитывается выручка. Необычно высокие и низкие цены отмечаются по границам 1-го и 99-го процентилей внутри категории — они остаются в данных и могут быть отдельно проверены при анализе.


In [13]:
rees_prepared = rees_sales.copy()
rees_prepared["sale_date"] = pd.to_datetime(rees_prepared["sale_date"], utc=True, errors="coerce").dt.tz_convert(None).dt.floor("D")
rees_prepared["price"] = pd.to_numeric(rees_prepared["price"], errors="coerce")
rees_prepared["quantity"] = pd.to_numeric(rees_prepared["quantity"], errors="coerce")
rees_prepared["brand"] = (
    rees_prepared["brand"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
)

rees_prepared = rees_prepared.loc[
    rees_prepared["sale_date"].notna()
    & rees_prepared["category"].isin(EXPECTED_CATEGORIES)
    & rees_prepared["price"].gt(0)
    & rees_prepared["quantity"].gt(0)
].drop_duplicates().copy()

rees_price_bounds = rees_prepared.groupby("category")["price"].quantile([0.01, 0.99]).unstack()
rees_price_bounds.columns = ["price_p01", "price_p99"]
rees_prepared = rees_prepared.join(rees_price_bounds, on="category")
rees_prepared["price_outlier"] = ~rees_prepared["price"].between(
    rees_prepared["price_p01"], rees_prepared["price_p99"], inclusive="both"
)

rees_prepared["source"] = "REES46"
rees_prepared["source_record_id"] = (
    "REES46:" + rees_prepared["order_id"].astype(str) + ":" + rees_prepared["product_id"].astype(str)
)
rees_prepared["city"] = pd.Series(pd.NA, index=rees_prepared.index, dtype="string")
rees_prepared["revenue"] = rees_prepared["price"] * rees_prepared["quantity"]

display(rees_prepared.head())
display(rees_prepared.groupby("category").agg(
    строк=("quantity", "size"),
    единиц=("quantity", "sum"),
    выбросов_цены=("price_outlier", "sum"),
    медианная_цена=("price", "median"),
).round(2))


,sale_date,order_id,product_id,category_code,brand,price,quantity,category,month,revenue,price_p01,price_p99,price_outlier,source,source_record_id,city
0,2020-04-01,2348828166153306143,1515966223509104702,computers.notebook,hp,624.98,1,Ноутбуки,4,624.98,4.38,1951.37,False,REES46,REES46:2348828166153306143:1515966223509104702,<NA>
1,2020-04-01,2348828225603371531,2273948316532212462,electronics.smartphone,huawei,254.61,1,Смартфоны,4,254.61,5.56,1387.01,False,REES46,REES46:2348828225603371531:2273948316532212462,<NA>
2,2020-04-01,2348828212735246760,1515966223509088493,electronics.smartphone,apple,740.72,1,Смартфоны,4,740.72,5.56,1387.01,False,REES46,REES46:2348828212735246760:1515966223509088493,<NA>
3,2020-04-01,2348828072670658826,1515966223509106147,computers.notebook,hp,706.00,1,Ноутбуки,4,706.00,4.38,1951.37,False,REES46,REES46:2348828072670658826:1515966223509106147,<NA>
4,2020-04-01,2348828072670658826,1515966223509570200,computers.notebook,none,4.38,1,Ноутбуки,4,4.38,4.38,1951.37,False,REES46,REES46:2348828072670658826:1515966223509570200,<NA>


,строк,единиц,выбросов_цены,медианная_цена
category,,,,
Настольные компьютеры,1763,1765,31,39.33
Ноутбуки,52029,52049,872,449.05
Планшеты,14392,14397,263,208.31
Смартфоны,244030,244131,2270,208.08


### 4.2 Подготовка 1С

В исходной таблице 1С отрицательное количество означает возврат. Возвраты сохраняются в отдельной таблице `console_returns`, а в таблицу продаж включаются только строки с положительными количеством и ценой. Это предотвращает смешение покупок и возвратов при расчёте спроса.


In [14]:
console_stage = console_required.copy()
console_stage["source_record_id"] = "1C:" + console_stage.index.astype(str)
console_stage["sale_date"] = pd.to_datetime(console_stage["sale_date"], errors="coerce").dt.floor("D")
console_stage["price"] = pd.to_numeric(console_stage["price"], errors="coerce")
console_stage["quantity"] = pd.to_numeric(console_stage["quantity"], errors="coerce")
console_stage["brand"] = console_stage["brand"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
console_stage["city"] = console_stage["city"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

console_returns = console_stage.loc[console_stage["quantity"].lt(0)].copy()
console_prepared = console_stage.loc[
    console_stage["sale_date"].notna()
    & console_stage["category"].isin(EXPECTED_CATEGORIES)
    & console_stage["price"].gt(0)
    & console_stage["quantity"].gt(0)
].drop_duplicates().copy()

console_price_bounds = console_prepared.groupby("category")["price"].quantile([0.01, 0.99]).unstack()
console_price_bounds.columns = ["price_p01", "price_p99"]
console_prepared = console_prepared.join(console_price_bounds, on="category")
console_prepared["price_outlier"] = ~console_prepared["price"].between(
    console_prepared["price_p01"], console_prepared["price_p99"], inclusive="both"
)

console_prepared["source"] = "1C Predict Future Sales"
console_prepared["revenue"] = console_prepared["price"] * console_prepared["quantity"]

print("Отделено строк возвратов:", len(console_returns))
display(console_prepared.head())
display(console_prepared.groupby("category").agg(
    строк=("quantity", "size"),
    единиц=("quantity", "sum"),
    выбросов_цены=("price_outlier", "sum"),
    медианная_цена=("price", "median"),
).round(2))


Отделено строк возвратов: 155


,sale_date,price,category,quantity,city,brand,item_name,item_category_name,shop_name,source_record_id,price_p01,price_p99,price_outlier,source,revenue
0,2013-01-18,8490.0,Игровые приставки,1.0,Москва,Microsoft,Microsoft Xbox 360 4GB (RKB-00011),Игровые консоли - XBOX 360,"Москва ТРК ""Атриум""",1C:0,990.0,29990.0,False,1C Predict Future Sales,8490.0
1,2013-01-06,15990.0,Игровые приставки,1.0,Москва,Microsoft,Xbox 360 250 ГБ + Kinect bundle матовая+Dance Central 2+Kinect Sports,Игровые консоли - XBOX 360,"Москва ТК ""Буденовский"" (пав.К7)",1C:1,990.0,29990.0,False,1C Predict Future Sales,15990.0
2,2013-01-09,15990.0,Игровые приставки,1.0,Москва,Microsoft,Xbox 360 250 ГБ + Kinect bundle матовая+Dance Central 2+Kinect Sports,Игровые консоли - XBOX 360,"Москва ТК ""Буденовский"" (пав.К7)",1C:2,990.0,29990.0,False,1C Predict Future Sales,15990.0
3,2013-01-26,15990.0,Игровые приставки,1.0,Москва,Microsoft,Xbox 360 250 ГБ + Kinect bundle матовая+Dance Central 2+Kinect Sports,Игровые консоли - XBOX 360,"Москва ТК ""Буденовский"" (пав.К7)",1C:3,990.0,29990.0,False,1C Predict Future Sales,15990.0
4,2013-01-12,13990.0,Игровые приставки,1.0,Москва,Microsoft,Xbox 360 320 ГБ+Halo 4 Special Edition (S4K-00077),Игровые консоли - XBOX 360,"Москва ТК ""Буденовский"" (пав.К7)",1C:4,990.0,29990.0,False,1C Predict Future Sales,13990.0


,строк,единиц,выбросов_цены,медианная_цена
category,,,,
Игровые приставки,24615,32314.0,227,13990.0


## 5. Слияние данных

Слияние выполняется на двух уровнях.

1. Таблицы 1С уже соединены по ключам товара, категории и магазина в ячейке выше раздела 2.3.
2. Подготовленные транзакции REES46 и 1С приводятся к одинаковому набору столбцов и объединяются вертикально. Поле `source` не позволяет потерять происхождение строк.

Это не означает, что объёмы двух компаний можно складывать и трактовать как продажи одного магазина.


In [15]:
COMMON_COLUMNS = [
    "source", "source_record_id", "sale_date", "category", "price", "quantity",
    "brand", "city", "revenue", "price_outlier",
]

combined_sales = pd.concat(
    [rees_prepared[COMMON_COLUMNS], console_prepared[COMMON_COLUMNS]],
    ignore_index=True,
    sort=False,
)
combined_sales["year_month"] = combined_sales["sale_date"].dt.to_period("M").astype(str)

monthly_sales = (
    combined_sales
    .groupby(["source", "year_month", "category"], as_index=False)
    .agg(
        transactions=("quantity", "size"),
        units=("quantity", "sum"),
        revenue=("revenue", "sum"),
        median_price=("price", "median"),
        brands=("brand", "nunique"),
        price_outliers=("price_outlier", "sum"),
    )
)

display(combined_sales.head())
display(monthly_sales.head(10))
print("Строк в объединённой таблице:", len(combined_sales))


,source,source_record_id,sale_date,category,price,quantity,brand,city,revenue,price_outlier,year_month
0,REES46,REES46:2348828166153306143:1515966223509104702,2020-04-01,Ноутбуки,624.98,1.0,hp,<NA>,624.98,False,2020-04
1,REES46,REES46:2348828225603371531:2273948316532212462,2020-04-01,Смартфоны,254.61,1.0,huawei,<NA>,254.61,False,2020-04
2,REES46,REES46:2348828212735246760:1515966223509088493,2020-04-01,Смартфоны,740.72,1.0,apple,<NA>,740.72,False,2020-04
3,REES46,REES46:2348828072670658826:1515966223509106147,2020-04-01,Ноутбуки,706.00,1.0,hp,<NA>,706.00,False,2020-04
4,REES46,REES46:2348828072670658826:1515966223509570200,2020-04-01,Ноутбуки,4.38,1.0,none,<NA>,4.38,False,2020-04


,source,year_month,category,transactions,units,revenue,median_price,brands,price_outliers
0,1C Predict Future Sales,2013-01,Игровые приставки,832,874.0,10759301.22,13490.0,2,0
1,1C Predict Future Sales,2013-02,Игровые приставки,616,648.0,7584473.00,12990.0,2,0
2,1C Predict Future Sales,2013-03,Игровые приставки,613,632.0,7033067.40,12490.0,2,0
3,1C Predict Future Sales,2013-04,Игровые приставки,426,437.0,5100121.90,12490.0,2,0
4,1C Predict Future Sales,2013-05,Игровые приставки,367,388.0,4332466.65,11490.0,2,0
5,1C Predict Future Sales,2013-06,Игровые приставки,547,561.0,6245199.70,11990.0,2,0
6,1C Predict Future Sales,2013-07,Игровые приставки,574,589.0,6922928.95,12994.0,2,0
7,1C Predict Future Sales,2013-08,Игровые приставки,596,612.0,7029556.20,11990.0,2,0
8,1C Predict Future Sales,2013-09,Игровые приставки,748,805.0,9524803.50,12490.0,2,0
9,1C Predict Future Sales,2013-10,Игровые приставки,728,790.0,8976074.35,11500.0,2,0


Строк в объединённой таблице: 336829


### 5.1 Присоединение контекста из разнородных источников

XML, Wildberries и тексты не содержат историю продаж. Поэтому они агрегируются до уровня категории и присоединяются к месячным продажам только как справочные признаки: число URL в каталоге, число актуальных предложений и длина полученного текста. Такое соединение не подменяет продажи ассортиментом.


In [16]:
category_context = pd.DataFrame({"category": sorted(EXPECTED_CATEGORIES)})

if "xml_coverage" in globals() and not xml_coverage.empty:
    xml_context = xml_coverage.rename(columns={"url_count": "gix_xml_urls"})
    category_context = category_context.merge(
        xml_context[["category", "gix_xml_urls"]], on="category", how="left", validate="one_to_one"
    )

if "wb_products" in globals() and not wb_products.empty:
    wb_context = wb_products.groupby("category", as_index=False).agg(
        wb_products=("product_id", "nunique"),
        wb_brands=("brand", "nunique"),
        wb_median_price_rub=("price_rub", "median"),
        wb_median_feedbacks=("feedbacks", "median"),
    )
    category_context = category_context.merge(
        wb_context, on="category", how="left", validate="one_to_one"
    )
else:
    category_context["wb_products"] = pd.NA
    print("Wildberries не присоединён: в сохранённом запуске API вернул 403 и ноль товаров.")

if "gix_texts" in globals() and not gix_texts.empty:
    text_context = gix_texts.groupby("category", as_index=False).agg(
        gix_text_length=("text_length", "median")
    )
    category_context = category_context.merge(
        text_context, on="category", how="left", validate="one_to_one"
    )

prepared_analytic_table = monthly_sales.merge(
    category_context, on="category", how="left", validate="many_to_one"
)

display(category_context)
display(prepared_analytic_table.head(10))


Wildberries не присоединён: в сохранённом запуске API вернул 403 и ноль товаров.


,category,gix_xml_urls,wb_products,gix_text_length
0,Игровые приставки,60,<NA>,1500.0
1,Настольные компьютеры,55,<NA>,1500.0
2,Ноутбуки,214,<NA>,1500.0
3,Планшеты,615,<NA>,1500.0
4,Смартфоны,2152,<NA>,1500.0


,source,year_month,category,transactions,units,revenue,median_price,brands,price_outliers,gix_xml_urls,wb_products,gix_text_length
0,1C Predict Future Sales,2013-01,Игровые приставки,832,874.0,10759301.22,13490.0,2,0,60,<NA>,1500.0
1,1C Predict Future Sales,2013-02,Игровые приставки,616,648.0,7584473.00,12990.0,2,0,60,<NA>,1500.0
2,1C Predict Future Sales,2013-03,Игровые приставки,613,632.0,7033067.40,12490.0,2,0,60,<NA>,1500.0
3,1C Predict Future Sales,2013-04,Игровые приставки,426,437.0,5100121.90,12490.0,2,0,60,<NA>,1500.0
4,1C Predict Future Sales,2013-05,Игровые приставки,367,388.0,4332466.65,11490.0,2,0,60,<NA>,1500.0
5,1C Predict Future Sales,2013-06,Игровые приставки,547,561.0,6245199.70,11990.0,2,0,60,<NA>,1500.0
6,1C Predict Future Sales,2013-07,Игровые приставки,574,589.0,6922928.95,12994.0,2,0,60,<NA>,1500.0
7,1C Predict Future Sales,2013-08,Игровые приставки,596,612.0,7029556.20,11990.0,2,0,60,<NA>,1500.0
8,1C Predict Future Sales,2013-09,Игровые приставки,748,805.0,9524803.50,12490.0,2,0,60,<NA>,1500.0
9,1C Predict Future Sales,2013-10,Игровые приставки,728,790.0,8976074.35,11500.0,2,0,60,<NA>,1500.0


## Проверка результата

Финальные проверки подтверждают, что обязательные поля заполнены, цена и количество положительны, категории входят в область исследования, а происхождение каждой строки известно. Проверки оформлены как `assert`: при нарушении условия выполнение остановится с ошибкой.


In [17]:
quality_after = pd.DataFrame([
    quality_snapshot(rees_prepared, "REES46 после подготовки"),
    quality_snapshot(console_prepared, "1С после подготовки"),
    quality_snapshot(combined_sales, "Объединённые продажи"),
])
display(quality_after)

assert set(COMMON_COLUMNS).issubset(combined_sales.columns)
assert combined_sales[["source", "sale_date", "category", "price", "quantity"]].notna().all().all()
assert combined_sales["price"].gt(0).all()
assert combined_sales["quantity"].gt(0).all()
assert set(combined_sales["category"].unique()).issubset(EXPECTED_CATEGORIES)
assert category_context["category"].is_unique
assert prepared_analytic_table["source"].notna().all()

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
combined_sales.to_csv(PROCESSED_DIR / "combined_sales_prepared.csv", index=False)
monthly_sales.to_csv(PROCESSED_DIR / "monthly_sales_prepared.csv", index=False)
prepared_analytic_table.to_csv(PROCESSED_DIR / "monthly_sales_with_context.csv", index=False)
console_returns.to_csv(PROCESSED_DIR / "console_returns.csv", index=False)

print("Все проверки пройдены.")
print("Подготовленные файлы сохранены в:", PROCESSED_DIR.resolve())


,набор,строк,полных_дубликатов,пропусков_в_обязательных_полях,price_min,price_max,quantity_min,quantity_max
0,REES46 после подготовки,312214,0,0,0.02,9606.48,1.0,4.0
1,1С после подготовки,24615,0,0,8.00,42990.00,1.0,101.0
2,Объединённые продажи,336829,0,0,0.02,42990.00,1.0,101.0


Все проверки пройдены.
Подготовленные файлы сохранены в: /private/tmp/digital-tech-sales-analysis-review-20261030/data/processed


## Вывод по заданию 2.3

Данные подготовлены для дальнейшего анализа:

1. обязательные поля приведены к согласованным типам и названиям;
2. некорректные строки исключены, а возвраты 1С выделены отдельно;
3. ценовые выбросы отмечены без автоматического удаления;
4. REES46 и 1С приведены к общей схеме и объединены с сохранением источника;
5. создана месячная таблица для анализа сезонности;
6. сведения XML, JSON и текстов присоединяются только на допустимом уровне категории;
7. итоговые таблицы прошли автоматические проверки и сохранены в `data/processed`.

**Ограничения:** REES46 охватывает только апрель–ноябрь 2020 года; 1С относится к 2013–2015 годам; источники принадлежат разным компаниям. Поэтому абсолютные объёмы между источниками не складываются и не сравниваются без нормализации. В сохранённом запуске Wildberries вернул ошибку 403, поэтому его показатели остаются пропущенными, а не заменяются вымышленными значениями.

Для сдачи задания 2.3 следует указать диапазон ячеек от заголовка **«Задание 2.3. Подготовка и слияние данных»** до этого вывода включительно.
